In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import selfies as sf
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import math
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, DataStructs, rdmolops, Descriptors, rdMolDescriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
#import plotly.express as px
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
import re
import difflib
torch.cuda.empty_cache()

In [2]:
df = pd.read_csv('/data/home2/andrze06/projects/Molecular-Spectro-Latent-Modeling/data/smiles_selfies_full.csv')

In [3]:
# prepare dataset
df['tokens'] = df['selfies'].apply(lambda x: list(sf.split_selfies(x)))
vocab = sorted(set([tok for seq in df['tokens'] for tok in seq]))
#vocab = sorted(list(sf.get_semantic_robust_alphabet()))
PAD, SOS, EOS, MASK = "", "", "", "MASK"
vocab = [PAD, SOS, EOS, MASK] + vocab
vocab_size = len(vocab)

tok2id = {tok: idx for idx, tok in enumerate(vocab)}
id2tok = {idx: tok for tok, idx in tok2id.items()}

def molecule_tok2id(tokens, tok2id):
    return np.array([1] + [tok2id[t] for t in tokens] + [2])

df['token_ids'] = df['tokens'].apply(lambda toks: molecule_tok2id(toks, tok2id))
#df['lenghts'] = df['token_ids'].apply(len)

sequences = df['token_ids'].tolist()
max_len = max(len(seq) for seq in sequences)
padded_data = np.zeros((len(sequences), max_len), dtype=sequences[0].dtype)

for i, seq in enumerate(sequences):
    padded_data[i, :len(seq)] = seq

data = padded_data#[:100_000]
train_data, temp_data = train_test_split(data, test_size=0.2, random_state=42, shuffle=True)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, shuffle=True)

In [4]:
# utilities
@torch.no_grad()
def accuracy(model, loader, mode='eval', pad_id=0, device='cuda'):
    model.eval()

    total_correct = 0
    total_tok = 0
    total_seq = 0
    perfect = 0

    for x in loader:
        x = x.to(device)

        if mode == 'train':
            logits, _, _ = model(x, mode=mode)
            targets = x[:, 1:]

            pred = logits.argmax(dim=-1)

            mask = (targets != pad_id)

            correct = (pred == targets) & mask

            total_correct += correct.sum().item()
            total_tok += mask.sum().item()

            seq_correct = (correct.sum(dim=1) == mask.sum(dim=1))
            perfect += seq_correct.sum().item()
            total_seq += x.size(0)

        else:
            tokens, _, _ = model(x, mode='eval')

            for i, pred in enumerate(tokens):
                true = x[i]
                mask = (true != pad_id)
                true_len = mask.sum().item()

                if true_len == 0:
                    continue
                if len(pred) < true_len:
                    pad = torch.full((true_len - len(pred),), pad_id, device=pred.device, dtype=pred.dtype)
                    pred = torch.cat([pred, pad], dim=0)
                else:
                    pred = pred[:true_len]

                true = true[:true_len]
                correct = (pred == true)
                total_correct += correct.sum().item()
                total_tok += true_len
                perfect += int(correct.all())
                total_seq += 1

    return (
        total_correct / max(total_tok, 1),
        perfect / max(total_seq, 1)
    )
     

In [6]:
# model
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        pe = torch.zeros(max_len, hidden_size)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_size, 2).float() * (-math.log(10000.0) / hidden_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        if x.dim() == 3:
            B, T, _ = x.shape
        elif x.dim() == 2:
            B, T = x.shape
        if T <= self.pe.size(0):
            pe = self.pe[:T]  
        else:
            device = x.device
            H = self.hidden_size
            position = torch.arange(T, dtype=torch.float, device=device).unsqueeze(1)  
            div_term = torch.exp(torch.arange(0, H, 2, device=device).float() * (-math.log(10000.0)/H))
            pe = torch.zeros(T, H, device=device)
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.d = hidden_size // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_k = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_v = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_o = nn.Linear(hidden_size, hidden_size, bias=False)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, 2 * hidden_size),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(2 * hidden_size, hidden_size),
            nn.Dropout(p=0.1)
        )
        self.norm2 = nn.LayerNorm(hidden_size)

    def forward(self, q, k, v, pad_mask=None, causal=False):   # [B, T, H]
        B, T_q, H = q.shape
        _, T_v, _ = v.shape
        Q = self.W_q(q)     # [B, T, num_heads * H]
        K = self.W_k(k)
        Vx = self.W_v(v)
        Q = Q.view(B, self.num_heads, T_q, self.d) # [B, A, T, H]
        K = K.view(B, self.num_heads, T_v, self.d)
        V = Vx.view(B, self.num_heads, T_v, self.d)

        attn_logits = torch.einsum('baih,bajh->baij', Q, K)    # [B, A, T, H] @ [B, A, H, T] = [B, A, T, T]

        if pad_mask is not None:
            key_mask = pad_mask[:, None, None, :]  # [B,1,1,T_k]
            attn_logits = attn_logits.masked_fill(~key_mask, float('-inf'))

        attn = F.softmax(attn_logits / math.sqrt(self.d), dim=-1)

        if pad_mask is not None:
            query_mask = pad_mask[:, None, :, None]  # [B,1,T_q,1]
            attn = attn * query_mask.float()

        h = torch.einsum('baij,bajh->baih',attn, V)  # [B, A, T, H]
        h = h.view(B, T_q, H)  # [B, T, A*H]

        # soft XSA
        # Vn = F.normalize(Vx, dim=-1)
        # h = h - 0.8 * (h * Vn).sum(dim=-1, keepdim=True) * Vn
        
        h = q + self.W_o(h)     # [B, T, H]
        h = h + self.ff(self.norm2(h))
        h = self.norm1(h)
        return h

class MultiSlotPooling(nn.Module):
    def __init__(self, hidden_size, num_slots):
        super().__init__()
        self.queries = nn.Parameter(torch.empty(num_slots, hidden_size))
        nn.init.uniform_(self.queries, a=-1.0, b=1.0)        
        self.W_k = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_v = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, h, mask):
        # h: [B, T, D]
        # mask: [B, T]
        k = self.W_k(h)
        v = self.W_v(h)
        attn = torch.einsum("kd,btd->bkt", self.queries, k)
        attn = attn.masked_fill(~mask[:, None, :], -1e9)
        attn = F.softmax(attn, dim=-1) / math.sqrt(h.size(-1))
        slots = torch.einsum("bkt,btd->bkd", attn, v)
        return slots
    

class VaeTransformer(nn.Module):
    def __init__(self, vocab_size, hidden_size, latent_size, max_len, attn_heads=8, num_slots=8, encoder_layers=1, decoder_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_slots = num_slots
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.pos_encoder = PositionalEmbedding(max_len, hidden_size)
        self.conv_pos = nn.Conv1d(hidden_size, hidden_size, kernel_size=3, padding=1, groups=hidden_size)
        
        # Encoder
        self.pos_block = MultiHeadAttention(hidden_size, attn_heads)
        self.encoder_blocks = nn.ModuleList([MultiHeadAttention(hidden_size, attn_heads) for _ in range(encoder_layers)])
        self.pool = MultiSlotPooling(hidden_size, num_slots=num_slots)
        #self.slot_pos_encoder = nn.Parameter(torch.randn(1, num_slots, hidden_size))
        self.slots_mix = MultiHeadAttention(hidden_size, num_heads=num_slots)

        self.slot_gamma = nn.Parameter(torch.ones(1, num_slots, hidden_size))
        self.slot_beta = nn.Parameter(torch.zeros(1, num_slots, hidden_size))
        #nn.init.orthogonal_(self.slot_beta[0])

        # VAE heads
        self.slot_mu = nn.Linear(hidden_size, latent_size)
        self.slot_logvar = nn.Linear(hidden_size, latent_size)  
        
        self.slot_compress_mu = nn.Linear(hidden_size, latent_size // num_slots)
        self.slot_compress_logvar = nn.Linear(hidden_size, latent_size // num_slots)

        self.fc_mu = nn.Linear(num_slots * hidden_size, latent_size)
        self.fc_logvar = nn.Linear(num_slots * hidden_size, latent_size)
        
        # pooling in latent space with uncertanty
        self.latent_query = nn.Parameter(torch.randn(1, 1, latent_size))
        self.latent_key = nn.Linear(latent_size, latent_size)

        # Decoder
        self.max_len = max_len
        self.z_to_slot = nn.Linear(hidden_size // num_slots, hidden_size)
        #self.slot_pos_decoder = nn.Parameter(torch.randn(1, num_slots, hidden_size))
        self.decoder_embed = nn.Embedding(vocab_size, hidden_size)

        self.decoder_pos = PositionalEmbedding(max_len, hidden_size)

        self.z_to_memory = nn.Linear(latent_size, hidden_size)
        self.slots_to_memory = nn.Linear(latent_size // num_slots, hidden_size)

        self.decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_size,
            nhead=attn_heads,
            dim_feedforward=2 * hidden_size,
            batch_first=True
        )

        self.decoder_transformer = nn.TransformerDecoder(
            self.decoder_layer,
            num_layers=decoder_layers
        )
        # Output head
        self.fc_output = nn.Linear(hidden_size, vocab_size)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def causal_mask(self, T, device):
        return torch.triu(
            torch.ones(T, T, device=device),
            diagonal=1
        ).bool()
    
    def encode(self, x, mode=None):  
        B, _ = x.shape
        h = self.embedding(x)    # [B, T, H]
        fourier_pos_encoding = self.pos_encoder(x)
        conv_pos_encoding = self.conv_pos(h.transpose(1, 2)).transpose(1, 2)
        h = h + fourier_pos_encoding + conv_pos_encoding
        mask = (x != 0)
        
        for block in self.encoder_blocks:
            h = block(h, h, h, pad_mask=mask)
            
        h = h.masked_fill(~mask[:, :, None], 0.0)

        slots = self.pool(h, mask)
        slots = F.layer_norm(slots, slots.shape[-1:])
        #slots = slots * self.slot_gamma + self.slot_beta

        B, K, Z = slots.shape

        slots_mu = self.slot_mu(slots)
        slots_logvar = self.slot_logvar(slots)

        # Latent pool from slots with uncertanty
        q = F.normalize(self.latent_query.expand(B, 1, Z), dim=-1)
        k = F.normalize(self.latent_key(slots_mu), dim=-1)

        logits = torch.einsum("bqz,bkz->bqk", q, k)

        confidence = -torch.logsumexp(slots_logvar, dim=-1).unsqueeze(1) #-slots_logvar.mean(dim=-1).unsqueeze(1)
        confidence_scale = 0.5

        logits = logits + confidence_scale * confidence
        attn = torch.softmax(logits / 0.5, dim=-1)

        mu = torch.einsum("bqk,bkz->bqz", attn, slots_mu).squeeze(1)

        var = torch.exp(slots_logvar)
        var_agg = torch.einsum("bqk,bkz->bqz", attn, var).squeeze(1)
        logvar = torch.log(var_agg + 1e-8)
        
        if mode == "test":
            return mu, logvar, slots
        else:
            return mu, logvar

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu

    def decode(self, z, x_in=None, max_len=80, start_id=1, eos_id=2):
        B = z.size(0)
        device = z.device

        memory = self.z_to_memory(z).unsqueeze(1)

        if x_in is not None:
            x_emb = self.decoder_embed(x_in)
            x_emb = x_emb + self.decoder_pos(x_emb)

            T = x_emb.size(1)
            tgt_mask = self.causal_mask(T, device)

            h = self.decoder_transformer(
                tgt=x_emb,
                memory=memory,
                tgt_mask=tgt_mask
            )

            logits = self.fc_output(h)   # ✔️ [B, T, V]
            return logits

        else:
            tokens = torch.full((B, 1), start_id, dtype=torch.long, device=device)
            finished = torch.zeros(B, dtype=torch.bool, device=device)
            max_len = self.max_len * 2

            for _ in range(max_len):
                x_emb = self.decoder_embed(tokens)
                x_emb = x_emb + self.decoder_pos(x_emb)

                T = tokens.size(1)
                tgt_mask = self.causal_mask(T, device)

                h = self.decoder_transformer(
                    tgt=x_emb,
                    memory=memory,
                    tgt_mask=tgt_mask
                )

                logits_step = self.fc_output(h[:, -1])  # [B, V]

                next_token = torch.argmax(logits_step, dim=-1, keepdim=True)

                next_token = torch.where(
                    finished.unsqueeze(1),
                    torch.full_like(next_token, eos_id),
                    next_token
                )

                tokens = torch.cat([tokens, next_token], dim=1)

                finished |= (next_token.squeeze(1) == eos_id)

                if finished.all():
                    break

            return tokens
            
    def forward(self, x, mode='eval'):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)

        if mode == 'train':
            x_in = x[:, :-1]

            logits = self.decode(z, x_in=x_in)

            return logits, mu, logvar
    

        if mode == "eval":
            tokens = self.decode(z, x_in=None)

            return tokens, mu, logvar 
        
        if mode == "test":
            mu, logvar, slots = self.encode(x, mode="test")
            z = self.reparameterize(mu, logvar)

            tokens = self.decode(z, x_in=None)

            return tokens, mu, logvar, slots


def vae_loss(logits, targets, mu, logvar, beta=0.01, pad_id=0):
    B, T, V = logits.shape

    logits = logits.reshape(-1, V)
    targets = targets.reshape(-1)

    rec_loss = F.cross_entropy(
        logits,
        targets,
        ignore_index=pad_id
    )

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    return rec_loss + beta * kl, rec_loss, kl

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

hidden_size = 256
attention_heads = 8
encoder_layers = 3
decoder_layers = 2
latent_size = 256

model = VaeTransformer(
    vocab_size,
    hidden_size,
    latent_size,
    max_len,
    attention_heads,
    encoder_layers=encoder_layers,
    decoder_layers=decoder_layers
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

beta = 0.01
epochs = 300
batch_size = 256
mask_prob = 0.05
corruption = False
max_beta = 0.03
cycle_lenght = 15

history = []
best_acc = 0

train_loader = DataLoader(train_data, batch_size, shuffle=True, num_workers=8)
val_loader = DataLoader(val_data, batch_size, shuffle=False, num_workers=8)

for epoch in range(1, epochs + 1):

    model.train()

    total_loss = 0
    total_rec = 0
    total_kl = 0

    pbar = tqdm(train_loader, dynamic_ncols=True, leave=False)

    # KL annealing
    # if epoch < 20:
    #     #beta = 0
    #     corruption = False
    # else:
    #     #beta = min(0.5, epoch / 60)
    #     corruption = False
    beta = (epoch % cycle_lenght / cycle_lenght) * max_beta

    for x in pbar:
        x = x.to(device)

        # input corruption
        if corruption:
            prob = torch.rand_like(x.float())

            mask = prob < mask_prob
            rand = (prob >= mask_prob) & (prob < mask_prob + 0.1)
            keep = (prob >= mask_prob + 0.1) & (prob < mask_prob + 0.2)
            
            x_corrupt = x.clone()
            x_corrupt[mask] = 3 # mask_token_id
            x_corrupt[rand] = torch.randint(0, vocab_size, size=x[rand].shape, device=x.device)

            pad_mask = (x == 0)
            x_corrupt[pad_mask] = 0

            logits, mu, logvar = model(x_corrupt, mode='train')
        else:
            logits, mu, logvar = model(x, mode='train')

        targets = x[:, 1:]

        loss, rec, kl = vae_loss(
            logits, targets, mu, logvar, beta=beta
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_rec += rec.item()
        total_kl += kl.item()

        pbar.set_postfix({
            "rec": f"{rec.item():.3f}",
            "kl": f"{kl.item():.3f}",
            "tot": f"{loss.item():.3f}",
        })

    model.eval()
    
    val_loss_total = 0
    with torch.no_grad():
        for x in val_loader:
            x = x.to(device)

            logits, mu, logvar = model(x, mode='train')
            targets = x[:, 1:]

            val_loss, val_rec, val_kl = vae_loss(
                logits, targets, mu, logvar, beta=beta
            )

            val_loss_total += val_loss.item()


    token_acc, seq_acc = accuracy(model, val_loader, mode='train')

    total_loss /= len(train_loader)
    total_rec /= len(train_loader)
    total_kl /= len(train_loader)
    val_loss_total /= len(val_loader)

    history.append((total_loss, total_rec, total_kl, val_loss_total))

    print(
        f"Epoch: {epoch:03d} | "
        f"loss={total_loss:.4f} | rec={total_rec:.4f} | kl={total_kl:.4f} | "
        f"val={val_loss_total:.4f} | "
        f"token_acc={token_acc*100:.2f}% | seq_acc={seq_acc*100:.2f}%"
    )

    if seq_acc > best_acc:
        best_acc = seq_acc

        ckpt = {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "history": history,
            "vocab_size": vocab_size
        }

        torch.save(
            ckpt,
            "/data/home2/andrze06/projects/Molecular-Spectro-Latent-Modeling/ProbeVAE/trained-models/H256-L256-3E-2D.pt"
        )

        print("saved")

Epoch: 001 | loss=1.0371 | rec=1.0316 | kl=2.7293 | val=0.4052 | token_acc=86.26% | seq_acc=9.00%
saved


Epoch: 002 | loss=0.4041 | rec=0.3914 | kl=3.1828 | val=0.2013 | token_acc=93.48% | seq_acc=30.74%
saved


Epoch: 003 | loss=0.2649 | rec=0.2473 | kl=2.9289 | val=0.1287 | token_acc=96.15% | seq_acc=47.39%
saved


Epoch: 004 | loss=0.2011 | rec=0.1798 | kl=2.6623 | val=0.0972 | token_acc=97.41% | seq_acc=58.30%
saved


Epoch: 005 | loss=0.1651 | rec=0.1411 | kl=2.4051 | val=0.0813 | token_acc=98.00% | seq_acc=64.92%
saved


...

Epoch: 065 | loss=0.0251 | rec=0.0153 | kl=0.9842 | val=0.0126 | token_acc=99.91% | seq_acc=97.21%
saved


Epoch: 066 | loss=0.0267 | rec=0.0154 | kl=0.9403 | val=0.0142 | token_acc=99.90% | seq_acc=97.09%


Epoch: 067 | loss=0.0283 | rec=0.0156 | kl=0.9032 | val=0.0159 | token_acc=99.90% | seq_acc=97.08%


Epoch: 068 | loss=0.0296 | rec=0.0157 | kl=0.8742 | val=0.0171 | token_acc=99.90% | seq_acc=96.89%


KeyboardInterrupt: 